# Ionic conductivity of aqueous sodium acetate (2/3): NVT production run

This notebook continues from the final state of `01_npt_equilibration.ipynb`
and performs a 1 ns production run in the NVT ensemble.
It does not contain any analysis, so that it can also be executed as a job without supervision,
as explained in `../03_noninteractive/01_noninteractive_notebook_on_hpc.ipynb`.
This is useful when you increase the length of the simulation.
The results are analyzed in `03_conductivity_analysis.ipynb`.

While the simulation is running, you can follow its progress in a terminal with
`tail -f scalars_nvt.csv`.
The last columns show the progress, the speed (in ns/day) and the estimated remaining time.

For a quick test, reduce `nstep_nvt` in the parameter cell below.

**Thermostat.**
For the computation of transport properties, the thermostat should perturb the dynamics as little as possible.
We therefore use a Nosé-Hoover chain thermostat with a relaxation time of 2 ps,
which is long compared to the timescales of the charge current fluctuations.
In contrast to the Langevin thermostat used for the equilibration,
the Nosé-Hoover thermostat is deterministic and global:
it rescales all velocities with the same factor,
instead of adding friction and random forces to every atom.
The volume is fixed at the final volume of the NpT run.
For this system, the volume fluctuations in the NpT ensemble are less than 1%,
so this choice has a negligible effect on the result.

**Outputs.**

- `traj_ions.dcd`: the positions of one atom per ion every 100 fs,
  for the computation of the charge current.
  For sodium, this is the Na⁺ ion itself and for acetate, the carboxylate carbon atom.
  These positions are *not* wrapped back into the simulation cell,
  so that the displacements of the ions are continuous.
- `traj_nvt.dcd`: snapshots of the full system every 10 ps,
  wrapped into the simulation cell for visualization.
- `final.pdb`: the final structure, also wrapped into the simulation cell.
- `scalars_nvt.csv`: scalar properties every 1 ps,
  including the conserved quantity of the Nosé-Hoover integrator and the performance of the simulation.

In [ ]:
from sys import stdout

import openmm as mm
from openmm import app, unit

In [ ]:
temperature = 293.15 * unit.kelvin
timestep = 2 * unit.femtosecond
tau_thermostat = 2 * unit.picosecond
# 500_000 steps = 1 ns. For a quick test, use e.g. 100_000 steps (200 ps).
# (Much shorter runs are too short for the analysis in 03_conductivity_analysis.ipynb.)
nstep_nvt = 500_000
# Intervals (in steps) for the different outputs.
# If you change interval_ions or timestep,
# also update frame_interval in 03_conductivity_analysis.ipynb.
interval_ions = 25  # 25 steps = 50 fs
interval_snapshots = 5000  # 5000 steps = 10 ps
interval_scalars = 500  # 500 steps = 1 ps

## 1. Restart from the final NpT state

The `System` (force field), the topology and the state (positions, velocities and box vectors)
are all loaded from the outputs of the previous notebook.
The time and the step counter are reset to zero, so that the production run starts at $t=0$.

In [ ]:
pdb = app.PDBFile("init_nvt.pdb")
with open("init_system.xml") as f:
    system = mm.XmlSerializer.deserialize(f.read())
with open("init_nvt_state.xml") as f:
    state = mm.XmlSerializer.deserialize(f.read())

# On GPUs, use mixed instead of the default single precision.
# This considerably stabilizes the conserved quantity of the Nosé-Hoover integrator,
# at a small cost in speed. (Platforms that do support this are silently skipped.)
for name in "CUDA", "OpenCL":
    try:
        mm.Platform.getPlatformByName(name).setPropertyDefaultValue("Precision", "mixed")
    except mm.OpenMMException:
        pass

integrator = mm.NoseHooverIntegrator(temperature, 1 / tau_thermostat, timestep)
simulation = app.Simulation(pdb.topology, system, integrator)
print("Platform:", simulation.context.getPlatform().getName())
simulation.context.setState(state)
simulation.context.setTime(0)
simulation.currentStep = 0

## 2. Reporters

To compute the charge current, it is sufficient to follow one atom per ion,
because the ions remain intact and every atom of an ion has the same average velocity.

The `StateDataReporter` of OpenMM does not report the conserved quantity of the Nosé-Hoover integrator.
This is the total energy of the system plus the energy of the heat bath.
It should remain nearly constant during the simulation, which is a useful check of the time step.
The small class below adds it as an extra column to the output.

In [ ]:
class NoseHooverReporter(app.StateDataReporter):
    """A StateDataReporter with an extra column for the conserved quantity."""

    def _constructHeaders(self):
        return [*super()._constructHeaders(), "Conserved Energy (kJ/mole)"]

    def _constructReportValues(self, simulation, state):
        values = super()._constructReportValues(simulation, state)
        conserved = (
            state.getPotentialEnergy()
            + state.getKineticEnergy()
            + simulation.integrator.computeHeatBathEnergy()
        )
        return [*values, conserved.value_in_unit(unit.kilojoule_per_mole)]

In [ ]:
# One atom per ion: Na+ and the carboxylate carbon (C2) of acetate.
ion_indices = [
    atom.index
    for atom in pdb.topology.atoms()
    if atom.residue.name == "NA" or (atom.residue.name == "ACT" and atom.name == "C2")
]
print("Number of ions:", len(ion_indices))

simulation.reporters.append(
    app.DCDReporter(
        "traj_ions.dcd", interval_ions, enforcePeriodicBox=False, atomSubset=ion_indices
    )
)
simulation.reporters.append(
    app.DCDReporter("traj_nvt.dcd", interval_snapshots, enforcePeriodicBox=True)
)
simulation.reporters.append(
    NoseHooverReporter(
        "scalars_nvt.csv",
        interval_scalars,
        time=True,
        potentialEnergy=True,
        kineticEnergy=True,
        temperature=True,
        progress=True,
        speed=True,
        remainingTime=True,
        totalSteps=nstep_nvt,
    )
)
# Progress on screen, at most 20 lines.
simulation.reporters.append(
    app.StateDataReporter(
        stdout,
        max(nstep_nvt // 20, 1),
        step=True,
        temperature=True,
        progress=True,
        remainingTime=True,
        speed=True,
        totalSteps=nstep_nvt,
    )
)

## 3. Production run

In [ ]:
simulation.step(nstep_nvt)

state = simulation.context.getState(getPositions=True, enforcePeriodicBox=True)
with open("final.pdb", "w") as f:
    app.PDBFile.writeFile(simulation.topology, state.getPositions(), f)